# Projected Separation Analysis

This notebook analyzes the projected separations of dual nuclei candidates
detected by the YOLOv11x-OBB model.

The notebook includes:

- Catalog inspection
- Descriptive statistics
- Histogram visualization
- Angular separation analysis
- Physical separation analysis

The catalog was generated using the projected separation pipeline
implemented in this repository.

In [2]:
import os
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [ ]:
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True

In [ ]:
CATALOG_PATH = (
    "../results/finalModel/"
    "projected_separations.csv"
)

df = pd.read_csv(CATALOG_PATH)

logging.info(f"Loaded catalog with {len(df)} rows.")

NameError: name 'pd' is not defined

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df[["dist_arcsec", "dist_kpc"]].describe()

In [ ]:
def compute_statistics(values, bins=80):

    stats = {}

    stats["N"] = len(values)

    stats["mean"] = values.mean()

    stats["median"] = values.median()

    stats["std"] = values.std()

    stats["min"] = values.min()

    stats["max"] = values.max()

    # Mode
    mode_vals = values.mode()

    stats["mode"] = (
        mode_vals.iloc[0]
        if not mode_vals.empty
        else None
    )

    stats["mode_frequency"] = (
        (values == stats["mode"]).sum()
        if stats["mode"] is not None
        else 0
    )

    # Percentiles
    stats["p25"] = values.quantile(0.25)

    stats["p75"] = values.quantile(0.75)

    # Histogram analysis
    counts, bin_edges = np.histogram(
        values,
        bins=bins
    )

    max_bin_index = np.argmax(counts)

    stats["max_bin_count"] = counts[max_bin_index]

    stats["max_bin_range"] = (
        bin_edges[max_bin_index],
        bin_edges[max_bin_index + 1]
    )

    return stats

In [ ]:
def plot_histogram(
    data,
    column_name,
    bins=80,
    save_dir="../docs/figures"
):

    if column_name not in data.columns:

        logging.error(
            f"Column '{column_name}' not found."
        )

        return

    os.makedirs(save_dir, exist_ok=True)

    values = data[column_name].dropna()

    stats = compute_statistics(
        values,
        bins
    )

    plt.figure(figsize=(8, 5))

    plt.hist(
        values,
        bins=bins,
        color="#A37FD3",
        edgecolor="white",
        linewidth=0.8,
        alpha=0.9
    )

    xlabel = (
        "Projected Separation (Arcseconds)"
        if column_name == "dist_arcsec"
        else "Projected Separation (kpc)"
    )

    plt.xlabel(xlabel)

    plt.ylabel("Count")

    # Statistics annotation
    text = (
        f"Mean = {stats['mean']:.3f}\n"
        f"Median = {stats['median']:.3f}\n"
        f"Std = {stats['std']:.3f}\n"
        f"Min = {stats['min']:.3f}\n"
        f"Max = {stats['max']:.3f}"
    )

    plt.text(
        0.97,
        0.97,
        text,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(
            facecolor="white",
            alpha=0.8
        )
    )

    plt.tight_layout()

    output_path = os.path.join(
        save_dir,
        f"{column_name}_histogram.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    logging.info(
        f"Saved histogram:\n{output_path}"
    )

    return stats

In [ ]:
arcsec_stats = plot_histogram(
    df,
    "dist_arcsec",
    bins=80
)

In [ ]:
stats_df = pd.DataFrame(
    [arcsec_stats, kpc_stats],
    index=["dist_arcsec", "dist_kpc"]
)

stats_df

In [ ]:
stats_output_path = (
    "../docs/figures/"
    "statistics_summary.csv"
)

stats_df.to_csv(
    stats_output_path,
    index=True
)

logging.info(
    f"Statistics saved:\n{stats_output_path}"
)

# Summary

This notebook provides exploratory analysis and visualization of
the projected separations derived from YOLOv11x-OBB detections.

The generated plots and statistics can be used for:
- paper figures
- catalog validation
- exploratory data analysis (EDA)
- reproducibility checks